In [ ]:
!pip install -q transformers requests sentence-transformers

import os, getpass
if 'HF_TOKEN' not in os.environ:
    os.environ['HF_TOKEN'] = getpass.getpass('HF token (read scope): ')

In [ ]:
import os, requests, time

HF_TOKEN = os.environ['HF_TOKEN']

def hf_zero_shot_api(text, labels):
    try:
        r = requests.post(
            'https://api-inference.huggingface.co/models/facebook/bart-large-mnli',
            headers={'Authorization': f'Bearer {HF_TOKEN}'},
            json={'inputs': text, 'parameters': {'candidate_labels': labels}},
            timeout=5)
        return r.json()
    except Exception as e:
        return {"error": "Network/Firewall blocked connection to Hugging Face API."}

resumes = [
    'Built React dashboards for 3 startups',
    'Implemented Spring Boot microservices in Java for fintech app',
    'Trained CNN for image classification using PyTorch, 87% accuracy',
    'Cleaned 100k row dataset using pandas + plotted in seaborn for monthly reports',
    'Wrote SQL queries against PostgreSQL, optimised 3 slow queries by 10x',
]
labels = ['frontend dev', 'backend dev', 'data analyst', 'ML engineer']

start = time.time()
for r in resumes:
    result = hf_zero_shot_api(r, labels)
    if 'error' in result:
        print(f'  Error skipped: {result["error"]}')
        break # Exit early so your console stays clean
    else:
        print(f'  {r[:50]:50} -> {result["labels"][0]} ({result["scores"][0]:.2f})')
print(f'\nAPI execution attempted.')

In [ ]:
from transformers import pipeline

classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

start = time.time()
for r in resumes:
    res = classifier(r, candidate_labels=labels)
    print(f'  {r[:50]:50} -> {res["labels"][0]} ({res["scores"][0]:.2f})')
print(f'\nLocal time (after download): {time.time()-start:.2f}s')

In [ ]:
sentiment = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')

answers = [
    'I really enjoyed working on the team and shipped 3 features.',
    'I was the only one writing code; everyone else was slow.',
    'I learned a lot from my mentor and grew technically.',
    "I had to redo most of my teammate's work because it was wrong.",
    'My internship was great — would recommend it to anyone.',
]

print('Sentiment scores:')
for a in answers:
    result = sentiment(a)[0]
    label = result['label']
    score = result['score']
    print(f'  [{label} {score:.2f}] {a[:60]}')


In [ ]:
import time

def time_call(fn, n_runs=3):
    times = []
    for _ in range(n_runs):
        start = time.time()
        fn()
        times.append(time.time() - start)
    return min(times), sum(times)/len(times)

def call_api():
    hf_zero_shot_api('Built React dashboards', ['frontend dev', 'backend dev'])
api_min, api_avg = time_call(call_api)

def call_local():
    classifier('Built React dashboards', candidate_labels=['frontend dev', 'backend dev'])
local_min, local_avg = time_call(call_local)

print(f'Inference timing comparison (3 runs each, after warm-up):')
print(f'  API:   min {api_min:.2f}s | avg {api_avg:.2f}s')
print(f'  Local: min {local_min:.2f}s | avg {local_avg:.2f}s')